# Compare Training Runs With The ACDC Paper

This notebook loads every experiment in `runs/`, summarizes training behavior, and compares the best validation Dice values with the results reported in *An Exploration of 2D and 3D Deep Learning Techniques for Cardiac MR Image Segmentation*.

Important comparison note: the local training logs report slice-level validation metrics from the project split. The paper reports volume-level challenge metrics at end-diastole (ED) and end-systole (ES), including postprocessing and surface-distance measures. The Dice comparisons below are therefore useful for orientation, not a strict reproduction claim.


In [ ]:
from pathlib import Path
import json
import math

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 80)
pd.set_option("display.precision", 4)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RUNS_ROOT = PROJECT_ROOT / "runs"
PAPER_PATH = PROJECT_ROOT / "papers" / "An-Exploration-of-2D-and-3D-Deep-Learning.pdf"

print(f"Project root: {PROJECT_ROOT}")
print(f"Runs root: {RUNS_ROOT}")
print(f"Paper: {PAPER_PATH.relative_to(PROJECT_ROOT) if PAPER_PATH.exists() else 'not found'}")


## Load Local Experiment Logs

In [ ]:
CLASS_NAMES = {
    "0": "Background",
    "1": "RV",
    "2": "Myo",
    "3": "LV",
}

MODEL_NAME_RULES = [
    ("fcn8", "FCN-8"),
    ("unet2d_modified", "2D U-Net (mod.)"),
    ("unet2d", "2D U-Net"),
    ("unet3d", "3D U-Net (mod.)"),
]


def infer_model_name(run_name):
    for token, model_name in MODEL_NAME_RULES:
        if run_name.startswith(token):
            return model_name
    return run_name


def load_config(config_path):
    if not config_path.exists():
        return {}
    return json.loads(config_path.read_text())


def load_runs(runs_root):
    frames = []
    configs = {}
    for metrics_path in sorted(runs_root.glob("*/metrics.csv")):
        run_dir = metrics_path.parent
        run_name = run_dir.name
        metrics = pd.read_csv(metrics_path)
        metrics["run"] = run_name
        metrics["model"] = infer_model_name(run_name)
        frames.append(metrics)
        configs[run_name] = load_config(run_dir / "config.json")

    if not frames:
        raise RuntimeError(f"No metrics.csv files found under {runs_root}")

    return pd.concat(frames, ignore_index=True), configs


metrics, configs = load_runs(RUNS_ROOT)
print(f"Loaded {metrics['run'].nunique()} runs and {len(metrics)} epoch rows.")
display(metrics.head())


## Run Configuration Overview

In [ ]:
def config_value(config, key, default=None):
    return config.get("args", {}).get(key, config.get(key, default))

config_rows = []
for run, config in configs.items():
    config_rows.append({
        "run": run,
        "model": infer_model_name(run),
        "epochs_configured": config_value(config, "epochs"),
        "batch_size": config_value(config, "batch_size"),
        "learning_rate": config_value(config, "learning_rate"),
        "loss": config_value(config, "loss", "cross_entropy"),
        "device": config.get("device"),
        "train_files": config.get("num_train_files"),
        "val_files": config.get("num_val_files"),
        "combined_run": "combined_from" in config,
        "weights": config_value(config, "weights"),
    })

config_summary = pd.DataFrame(config_rows).sort_values(["model", "run"])
display(config_summary)


## Best Epoch Summary

In [ ]:
best_indices = metrics.groupby("run")["val_mean_foreground_dice"].idxmax()
best_rows = metrics.loc[best_indices].copy().set_index("run")
last_rows = metrics.sort_values("epoch").groupby("run", as_index=False).tail(1).set_index("run")

summary_rows = []
for run, best in best_rows.iterrows():
    last = last_rows.loc[run]
    config = configs.get(run, {})
    summary_rows.append({
        "run": run,
        "model": best["model"],
        "logged_epochs": int(metrics.loc[metrics["run"] == run, "epoch"].max()),
        "best_epoch": int(best["epoch"]),
        "best_val_mean_fg_dice": best["val_mean_foreground_dice"],
        "final_val_mean_fg_dice": last["val_mean_foreground_dice"],
        "best_val_loss": best["val_loss"],
        "best_val_pixel_accuracy": best["val_pixel_accuracy"],
        "best_val_RV_dice": best.get("val_dice_class_1", math.nan),
        "best_val_Myo_dice": best.get("val_dice_class_2", math.nan),
        "best_val_LV_dice": best.get("val_dice_class_3", math.nan),
        "loss": config_value(config, "loss", "cross_entropy"),
        "learning_rate": config_value(config, "learning_rate"),
        "batch_size": config_value(config, "batch_size"),
    })

run_summary = pd.DataFrame(summary_rows).sort_values("best_val_mean_fg_dice", ascending=False).reset_index(drop=True)
display(run_summary.style.format({
    "best_val_mean_fg_dice": "{:.3f}",
    "final_val_mean_fg_dice": "{:.3f}",
    "best_val_loss": "{:.4f}",
    "best_val_pixel_accuracy": "{:.3f}",
    "best_val_RV_dice": "{:.3f}",
    "best_val_Myo_dice": "{:.3f}",
    "best_val_LV_dice": "{:.3f}",
    "learning_rate": "{:.1e}",
}))


## Training Curves

In [ ]:
curve_runs = run_summary.loc[~run_summary["run"].str.contains("5epoch", case=False), "run"].tolist()
curve_data = metrics[metrics["run"].isin(curve_runs)].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharex=False)

sns.lineplot(data=curve_data, x="epoch", y="val_mean_foreground_dice", hue="run", ax=axes[0])
axes[0].set_title("Validation Mean Foreground Dice")
axes[0].set_ylabel("Dice")
axes[0].set_ylim(0, 1)
axes[0].legend(title="Run", bbox_to_anchor=(1.02, 1), loc="upper left")

sns.lineplot(data=curve_data, x="epoch", y="val_loss", hue="run", ax=axes[1], legend=False)
axes[1].set_title("Validation Loss")
axes[1].set_ylabel("Loss")

plt.tight_layout()


## Per-Class Best Validation Dice

In [ ]:
per_class_cols = ["best_val_RV_dice", "best_val_Myo_dice", "best_val_LV_dice"]
per_class = run_summary[["run", "model", *per_class_cols]].melt(
    id_vars=["run", "model"],
    var_name="structure",
    value_name="dice",
)
per_class["structure"] = per_class["structure"].map({
    "best_val_RV_dice": "RV",
    "best_val_Myo_dice": "Myo",
    "best_val_LV_dice": "LV",
})

plt.figure(figsize=(14, 6))
sns.barplot(data=per_class, x="run", y="dice", hue="structure")
plt.xticks(rotation=45, ha="right")
plt.ylim(0, 1)
plt.title("Best Local Validation Dice By Structure")
plt.ylabel("Dice")
plt.xlabel("")
plt.tight_layout()


## Paper Reference Values

The values below are transcribed from the paper tables. Table 1 reports modified 2D U-Net loss-function results averaged over both cardiac phases. Table 2 reports architecture results by structure and phase.


In [ ]:
paper_loss = pd.DataFrame([
    {"model": "2D U-Net (mod.)", "loss": "cross_entropy", "structure": "LV", "paper_dice": 0.950, "paper_std": 0.029},
    {"model": "2D U-Net (mod.)", "loss": "cross_entropy", "structure": "RV", "paper_dice": 0.891, "paper_std": 0.084},
    {"model": "2D U-Net (mod.)", "loss": "cross_entropy", "structure": "Myo", "paper_dice": 0.888, "paper_std": 0.031},
    {"model": "2D U-Net (mod.)", "loss": "weighted_cross_entropy", "structure": "LV", "paper_dice": 0.950, "paper_std": 0.036},
    {"model": "2D U-Net (mod.)", "loss": "weighted_cross_entropy", "structure": "RV", "paper_dice": 0.893, "paper_std": 0.083},
    {"model": "2D U-Net (mod.)", "loss": "weighted_cross_entropy", "structure": "Myo", "paper_dice": 0.899, "paper_std": 0.032},
    {"model": "2D U-Net (mod.)", "loss": "dice", "structure": "LV", "paper_dice": 0.944, "paper_std": 0.051},
    {"model": "2D U-Net (mod.)", "loss": "dice", "structure": "RV", "paper_dice": 0.843, "paper_std": 0.137},
    {"model": "2D U-Net (mod.)", "loss": "dice", "structure": "Myo", "paper_dice": 0.891, "paper_std": 0.029},
])

architecture_values = {
    "FCN-8": {
        "LV": {"ED": (0.960, 0.018), "ES": (0.926, 0.061)},
        "RV": {"ED": (0.932, 0.025), "ES": (0.835, 0.100)},
        "Myo": {"ED": (0.869, 0.029), "ES": (0.890, 0.027)},
    },
    "2D U-Net": {
        "LV": {"ED": (0.965, 0.014), "ES": (0.937, 0.051)},
        "RV": {"ED": (0.936, 0.028), "ES": (0.838, 0.085)},
        "Myo": {"ED": (0.885, 0.027), "ES": (0.904, 0.029)},
    },
    "2D U-Net (mod.)": {
        "LV": {"ED": (0.966, 0.017), "ES": (0.935, 0.042)},
        "RV": {"ED": (0.934, 0.039), "ES": (0.852, 0.095)},
        "Myo": {"ED": (0.892, 0.027), "ES": (0.906, 0.034)},
    },
    "3D U-Net (mod.)": {
        "LV": {"ED": (0.939, 0.022), "ES": (0.905, 0.039)},
        "RV": {"ED": (0.888, 0.069), "ES": (0.781, 0.101)},
        "Myo": {"ED": (0.802, 0.053), "ES": (0.839, 0.066)},
    },
}

paper_arch_rows = []
for model, structures in architecture_values.items():
    for structure, phases in structures.items():
        for phase, (dice, std) in phases.items():
            paper_arch_rows.append({
                "model": model,
                "structure": structure,
                "phase": phase,
                "paper_dice": dice,
                "paper_std": std,
            })

paper_arch = pd.DataFrame(paper_arch_rows)
paper_arch_avg = paper_arch.groupby(["model", "structure"], as_index=False).agg(
    paper_phase_avg_dice=("paper_dice", "mean"),
    paper_phase_avg_std=("paper_std", "mean"),
)
paper_arch_model_avg = paper_arch_avg.groupby("model", as_index=False).agg(
    paper_mean_fg_dice=("paper_phase_avg_dice", "mean"),
)

print("Paper Table 1: loss function Dice")
display(paper_loss)
print("Paper Table 2: architecture Dice averaged over ED and ES")
display(paper_arch_avg)
print("Paper Table 2: mean over LV/RV/Myo after phase averaging")
display(paper_arch_model_avg.sort_values("paper_mean_fg_dice", ascending=False))


## Local Runs Versus Paper Architecture Results

In [ ]:
local_structure = per_class.rename(columns={"dice": "local_dice"}).copy()
comparison = local_structure.merge(paper_arch_avg, on=["model", "structure"], how="left")
comparison["delta_vs_paper_phase_avg"] = comparison["local_dice"] - comparison["paper_phase_avg_dice"]

comparison_table = comparison.sort_values(["model", "run", "structure"])
display(comparison_table.style.format({
    "local_dice": "{:.3f}",
    "paper_phase_avg_dice": "{:.3f}",
    "paper_phase_avg_std": "{:.3f}",
    "delta_vs_paper_phase_avg": "{:+.3f}",
}))

plt.figure(figsize=(14, 6))
plot_data = comparison.dropna(subset=["paper_phase_avg_dice"]).copy()
sns.barplot(data=plot_data, x="run", y="delta_vs_paper_phase_avg", hue="structure")
plt.axhline(0, color="black", linewidth=1)
plt.xticks(rotation=45, ha="right")
plt.title("Best Local Validation Dice Minus Paper Phase-Averaged Dice")
plt.ylabel("Dice difference")
plt.xlabel("")
plt.tight_layout()


## Best Local Model Compared With Paper Ranking

In [ ]:
best_per_model = run_summary.sort_values("best_val_mean_fg_dice", ascending=False).groupby("model", as_index=False).head(1)
model_comparison = best_per_model.merge(paper_arch_model_avg, on="model", how="left")
model_comparison["delta_vs_paper_mean_fg"] = model_comparison["best_val_mean_fg_dice"] - model_comparison["paper_mean_fg_dice"]
model_comparison = model_comparison.sort_values("best_val_mean_fg_dice", ascending=False)

display(model_comparison[[
    "model",
    "run",
    "best_epoch",
    "best_val_mean_fg_dice",
    "paper_mean_fg_dice",
    "delta_vs_paper_mean_fg",
    "loss",
    "learning_rate",
    "batch_size",
]].style.format({
    "best_val_mean_fg_dice": "{:.3f}",
    "paper_mean_fg_dice": "{:.3f}",
    "delta_vs_paper_mean_fg": "{:+.3f}",
    "learning_rate": "{:.1e}",
}))

combined = pd.concat([
    model_comparison[["model", "best_val_mean_fg_dice"]].rename(columns={"best_val_mean_fg_dice": "dice"}).assign(source="local best run"),
    paper_arch_model_avg.rename(columns={"paper_mean_fg_dice": "dice"}).assign(source="paper Table 2 avg"),
], ignore_index=True)

plt.figure(figsize=(10, 5))
sns.barplot(data=combined.dropna(subset=["dice"]), x="model", y="dice", hue="source")
plt.ylim(0, 1)
plt.title("Best Local Mean Foreground Dice vs Paper Architecture Average")
plt.ylabel("Mean Dice over LV/RV/Myo")
plt.xlabel("")
plt.tight_layout()


## Modified U-Net Loss Comparison

In [ ]:
local_loss_runs = run_summary[run_summary["model"] == "2D U-Net (mod.)"].copy()
local_loss_structure = per_class[per_class["run"].isin(local_loss_runs["run"])].merge(
    local_loss_runs[["run", "loss"]], on="run", how="left"
).rename(columns={"dice": "local_dice"})

loss_comparison = local_loss_structure.merge(
    paper_loss,
    on=["model", "loss", "structure"],
    how="left",
)
loss_comparison["delta_vs_paper_loss_table"] = loss_comparison["local_dice"] - loss_comparison["paper_dice"]

display(loss_comparison.sort_values(["loss", "run", "structure"]).style.format({
    "local_dice": "{:.3f}",
    "paper_dice": "{:.3f}",
    "paper_std": "{:.3f}",
    "delta_vs_paper_loss_table": "{:+.3f}",
}))


## Observations Template

Run the notebook, then use the tables above to fill in these notes:

- Best local run by validation mean foreground Dice: see `run_summary.iloc[0]`.
- Strongest local structure: compare RV/Myo/LV bars in the per-class plot.
- Biggest gap to the paper: sort `comparison_table` by `delta_vs_paper_phase_avg`.
- Most important caveat: local values are validation-slice Dice, while paper values are ED/ES volume-level challenge metrics with ASSD/HD, so exact parity is not expected.


In [ ]:
print("Best local run:")
display(run_summary.iloc[[0]])

print("Largest local shortfalls against paper phase-averaged Dice:")
display(
    comparison_table.dropna(subset=["delta_vs_paper_phase_avg"])
    .sort_values("delta_vs_paper_phase_avg")
    .head(10)
    [["run", "model", "structure", "local_dice", "paper_phase_avg_dice", "delta_vs_paper_phase_avg"]]
)
